# Example 3.4 / 1: Plain and formatted output

Companion notebook for **3.4 "print and printf"** in the AMPL lab Documentation, section "Plain and formatted output". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Plain and formatted output

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
param profit {PROD};
param Make {PROD} >= 0;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["Make"] = {"bands": 6000, "coils": 4000}

In [5]:
# run.py
ampl.eval('print "Total profit:", sum {p in PROD} profit[p] * Make[p];')
ampl.eval('printf {p in PROD} "%-8s %8.1f\\n", p, Make[p];')

Total profit: 270000
bands      6000.0
coils      4000.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param: profit Make :=
  bands 25 6000
  coils 30 4000;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
print "Total profit:", sum {p in PROD} profit[p] * Make[p];
printf {p in PROD} "%-8s %8.1f\n", p, Make[p];

Total profit: 270000
bands      6000.0
coils      4000.0
